# 01 - Thu thập dữ liệu đánh giá Laptop trên FPTShop
Notebook này crawl dữ liệu đánh giá từ FPTShop theo hướng HTTP thuần (requests), chuẩn hóa schema tương thích pipeline EDA và lưu CSV UTF-8.

In [ ]:
%pip install requests pandas

In [ ]:
import json
import re
import time
from datetime import datetime
from urllib.parse import urljoin, urlparse, parse_qsl, urlencode
import hashlib

import pandas as pd
import requests

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
    "Referer": "https://fptshop.com.vn/"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def with_query_param(url: str, key: str, value):
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    query[key] = str(value)
    return parsed._replace(query=urlencode(query)).geturl()

def collect_product_urls(search_url: str, max_pages: int = 20, max_products: int = 120, delay_seconds: float = 0.8):
    product_urls = []
    seen = set()

    for page in range(1, max_pages + 1):
        page_url = with_query_param(search_url, "page", page)
        response = SESSION.get(page_url, timeout=20)
        response.raise_for_status()
        html_text = response.text

        matches = re.findall(r'href="(https://fptshop\.com\.vn/may-tinh-xach-tay/[^"]+|/may-tinh-xach-tay/[^"]+)"', html_text)
        new_count = 0
        for href in matches:
            full_url = urljoin("https://fptshop.com.vn", href.split("?")[0].strip())
            if full_url in seen:
                continue
            seen.add(full_url)
            product_urls.append(full_url)
            new_count += 1
            if len(product_urls) >= max_products:
                break

        print(f"Trang {page}: +{new_count} URL sản phẩm, tổng {len(product_urls)}")

        if new_count == 0 or len(product_urls) >= max_products:
            break
        time.sleep(delay_seconds)

    return product_urls

def extract_product_code(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')
    match = re.search(r'"upc":\{"code":"([^"]+)"\}', normalized)
    if match:
        return match.group(1)

    match_alt = re.search(r'"sku"\s*:\s*"([0-9]{6,})"', normalized)
    if match_alt:
        return match_alt.group(1)

    return None

def extract_comment_payload(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    pattern = r'"comment":(\{.*?"totalCount":\d+.*?\})\s*,\s*"policyProduct"'
    match = re.search(pattern, normalized, flags=re.DOTALL)
    if not match:
        return {}

    comment_raw = match.group(1)
    try:
        return json.loads(comment_raw)
    except json.JSONDecodeError:
        return {}

def fetch_product_reviews(product_url: str):
    response = SESSION.get(product_url, timeout=20)
    response.raise_for_status()
    html_text = response.text

    item_code = extract_product_code(html_text)
    comment_payload = extract_comment_payload(html_text)

    review_items = comment_payload.get("items") or []
    total_count = comment_payload.get("totalCount", len(review_items))

    return item_code, review_items, total_count

def parse_rating_row(rating_row: dict, item_code: str, product_url: str, row_index: int):
    created_raw = (
        rating_row.get("createdAt")
        or rating_row.get("createAt")
        or rating_row.get("createdDate")
        or rating_row.get("created_time")
        or rating_row.get("time")
    )
    created_at = pd.to_datetime(created_raw, errors="coerce")
    created_at = created_at.isoformat() if pd.notna(created_at) else None

    user_id = (
        rating_row.get("userId")
        or rating_row.get("customerId")
        or rating_row.get("memberId")
        or rating_row.get("fullName")
        or "anonymous"
    )
    cmt_id = rating_row.get("id") or rating_row.get("commentId") or rating_row.get("_id")
    if cmt_id is None:
        fallback = f"{product_url}_{row_index}_{rating_row.get('content', '')}"
        cmt_id = hashlib.md5(fallback.encode("utf-8")).hexdigest()[:16]

    rating_star = rating_row.get("score") or rating_row.get("rating") or rating_row.get("star")
    comment = (
        rating_row.get("content")
        or rating_row.get("comment")
        or rating_row.get("commentContent")
        or ""
    )
    product_items = (
        rating_row.get("variantName")
        or rating_row.get("modelName")
        or rating_row.get("attributeValue")
        or ""
    )

    return {
        "review_id": f"{user_id}_{cmt_id}",
        "shop_id": "FPTShop",
        "item_id": item_code,
        "user_id": str(user_id),
        "rating_star": rating_star,
        "comment": str(comment).strip(),
        "created_at": created_at,
        "like_count": rating_row.get("like") or rating_row.get("helpful") or 0,
        "product_items": str(product_items).strip(),
        "source": "FPTShop"
    }

def crawl_product_reviews(product_url: str, max_reviews: int = 500, page_size: int = 50, delay_seconds: float = 1.2):
    item_code, page_reviews, total_count = fetch_product_reviews(product_url=product_url)

    reviews = []
    seen_review_ids = set()

    for idx, row in enumerate(page_reviews):
        if len(reviews) >= max_reviews:
            break

        parsed = parse_rating_row(row, item_code=item_code, product_url=product_url, row_index=idx)
        if not parsed["comment"]:
            continue
        if parsed["review_id"] in seen_review_ids:
            continue

        seen_review_ids.add(parsed["review_id"])
        reviews.append(parsed)

    print(f"{product_url} -> lấy {len(reviews)} review hiển thị / totalCount={total_count}")
    if total_count > len(page_reviews):
        print("Lưu ý: hiện chỉ thu được danh sách review hiển thị từ HTML, chưa phân trang sâu bằng endpoint nội bộ.")

    time.sleep(delay_seconds)
    return pd.DataFrame(reviews)

In [ ]:
search_url = "https://fptshop.com.vn/tim-kiem?s=laptop&sort=noi-bat&categories=may-tinh-xach-tay"
product_urls = collect_product_urls(
    search_url=search_url,
    max_pages=25,
    max_products=150,
    delay_seconds=0.8
)

display(product_urls[:10])
print(f"Tổng URL sản phẩm laptop đã thu thập: {len(product_urls)}")

if not product_urls:
    raise ValueError("Không lấy được danh sách product_urls từ trang tìm kiếm FPTShop.")

In [ ]:
all_frames = []
for url in product_urls:
    df_item = crawl_product_reviews(url, max_reviews=400, page_size=50, delay_seconds=1.2)
    if not df_item.empty:
        all_frames.append(df_item)

if not all_frames:
    raise RuntimeError("Không thu được dữ liệu. Kiểm tra lại URL hoặc thử chạy lại với mạng khác.")

df_raw = pd.concat(all_frames, ignore_index=True)
df_raw = df_raw.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

display(df_raw.head())
display(df_raw.tail())
print(f"Tổng số review thu được: {len(df_raw)}")

In [ ]:
output_path = "data/fptshop_laptop_raw.csv"
df_raw.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu dữ liệu thô vào: {output_path}")

## Ghi chú
FPTShop có thể thay đổi cấu trúc dữ liệu đánh giá theo từng thời điểm. Notebook hiện ưu tiên requests/http thuần và đọc review hiển thị trong HTML của trang sản phẩm; nếu số lượng review thực tế lớn hơn dữ liệu thu được, cần reverse endpoint nội bộ để phân trang sâu hơn.